# Chapter 2 – Working with Text Data
### `embeddings.ipynb` — Samuel Leonardo Albarracin Vergara
### Escuela Colombiana de Ingeniería Julio Garavito · Ingernieria de Sistemas 

Based on *Build a Large Language Model (From Scratch)* by Sebastian Raschka.  
Repository: https://github.com/rasbt/LLMs-from-scratch

---
Este notebook sigue el capítulo 2 del libro y documenta el pipeline completo de preparación de texto para un LLM: desde el texto crudo hasta los embeddings posicionales que alimentan la red neuronal.


In [ ]:
from importlib.metadata import version
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

---
## 🧠 Explicación 1 — ¿Por qué necesitamos embeddings?

Las redes neuronales no pueden procesar texto directamente, porque solo trabajan con números. Para resolver esto se usan embeddings, que convierten palabras o fragmentos de palabras en vectores numéricos. En ese espacio matemático, la distancia entre vectores refleja el significado: palabras con sentidos parecidos quedan más cerca entre sí.

Lo interesante es que estas representaciones aparecen durante el entrenamiento del modelo, sin que alguien las programe explícitamente. Al aprender a predecir el contexto de las palabras, el modelo descubre relaciones semánticas por sí solo. Por eso se pueden observar analogías como: king − man + woman ≈ queen. En esencia, los embeddings funcionan como un puente entre el lenguaje humano y los cálculos matemáticos del modelo.

En los sistemas agénticos, esto es fundamental. Un agente que elige herramientas, planifica tareas o recupera información lo hace usando el espacio de embeddings. Por eso, la calidad de estos vectores influye directamente en qué tan bien el agente entiende la intención del usuario y el contexto.


## 2.1 Understanding word embeddings

Los LLMs trabajan con embeddings en espacios de alta dimensión (miles de dimensiones). La figura conceptual muestra un espacio 2D para visualizar la idea: palabras semánticamente similares quedan cerca entre sí.


## 2.2 Tokenizing text


In [ ]:
import os

# El archivo ya debe estar en el mismo directorio que este notebook
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print(raw_text[:99])

In [ ]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

In [ ]:
result = re.split(r'([,.]|\s)', text)
result = [item for item in result if item.strip()]
print(result)

In [ ]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

In [ ]:
print(len(preprocessed))

## 2.3 Converting tokens into token IDs


In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

In [ ]:
vocab = {token: integer for integer, token in enumerate(all_words)}

In [ ]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

In [ ]:
tokenizer.decode(ids)

## 2.4 Adding special context tokens


In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}
print(len(vocab.items()))

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>"
            for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

In [ ]:
tokenizer.encode(text)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

---
## 🧠 Explicación 2 — Tokens especiales y vocabulario abierto

El tokenizador V1 tiene un problema importante: falla cuando encuentra palabras que no están en el vocabulario con el que fue entrenado. Esto es crítico en entornos reales, donde siempre aparecerán términos nuevos.

Para manejar estos casos existen tokens especiales como <|unk|> y <|endoftext|>, que actúan como información adicional dentro del propio texto:

<|unk|> indica que había una palabra desconocida. Así se mantiene la posición del texto sin que el sistema falle.

<|endoftext|> marca el final de un documento. Esto permite unir varios textos en una sola secuencia y que el modelo aprenda dónde termina uno y empieza otro.

En los sistemas agénticos, esto es especialmente importante porque los agentes procesan información proveniente de múltiples fuentes (herramientas, memoria, usuario). El token <|endoftext|> ayuda a mantener esos contextos separados y evita que el modelo mezcle información que no debería combinarse.


## 2.5 BytePair Encoding (BPE)


In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

In [ ]:
strings = tokenizer.decode(integers)
print(strings)

## 2.6 Data sampling with a sliding window


In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

In [ ]:
enc_sample = enc_text[50:]
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f"x: {x}")
print(f"y:      {y}")

In [ ]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

In [ ]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

In [ ]:
import torch
print("PyTorch version:", torch.__version__)

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, \
            "Number of tokenized inputs must at least be equal to max_length+1"

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset, batch_size=batch_size,
        shuffle=shuffle, drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
second_batch = next(data_iter)
print(second_batch)

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

---
## 🧠 Explicación 3 — Sliding Window: por qué el overlap importa

El sliding window  es el mecanismo para crear pares (input, target) a partir de texto continuo. stride controla cuánto avanza la ventana en cada paso:

- stride = max_length (sin overlap): Cada token aparece en exactamente una muestra. El modelo ve poco contexto de transición entre ventanas → puede fallar en patrones que cruzan los límites de ventana.
- stride < max_length (con overlap):Los mismos tokens aparecen en múltiples samples con diferente contexto. Esto actúa como **data augmentation**, exponiendo al modelo a más combinaciones de contexto.

Para LLMs, el overlap es especialmente valioso al inicio del entrenamiento cuando el modelo necesita aprender dependencias a largo plazo. El tradeoff es que más overlap → más samples → más tiempo de entrenamiento y riesgo de overfitting si el dataset es pequeño.


---
## Experimento — Efecto de max_length y stride en el número de samples

Variamos los parámetros y observamos cómo cambia la cantidad de samples generados.


In [ ]:
import tiktoken

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
enc_text = tokenizer.encode(raw_text)
total_tokens = len(enc_text)
print(f"Total BPE tokens en the-verdict.txt: {total_tokens}")

configs = [
    (4,   1,   "overlap máximo, cada token en 4 samples"),
    (4,   4,   "sin overlap, máx. eficiencia de datos"),
    (128, 64,  "50% overlap, buena cobertura"),
    (128, 128, "sin overlap con ventana grande"),
    (256, 128, "50% overlap con contexto largo"),
    (256, 256, "sin overlap con contexto largo"),
]

print()
print(f"  {'max_length':>12}  {'stride':>6}  {'samples':>9}  observación")
for ml, st, obs in configs:
    samples = len(list(range(0, total_tokens - ml, st)))
    print(f"  {ml:>12}  {st:>6}  {samples:>9}  {obs}")

### Análisis del experimento

**Fórmula general:**  `num_samples = floor((total_tokens - max_length) / stride)`

**Observaciones clave:**

1. **`max_length=4, stride=1` → 5141 samples.** El overlap máximo genera ~4× más samples que sin overlap. Cada token aparece como punto de inicio en múltiples ventanas. Útil para datasets pequeños, pero aumenta el tiempo de entrenamiento y puede causar overfitting.

2. **`max_length=4, stride=4` → 1286 samples.** Sin overlap: cada token aparece en exactamente una muestra. Entrenamiento más rápido pero el modelo pierde contexto en las transiciones entre ventanas.

3. **`max_length=256, stride=128` → 39 samples vs `max_length=256, stride=256` → 20 samples.** Con un dataset pequeño como *The Verdict* (~5K tokens), usar contextos largos deja muy pocas muestras. GPT-2 usa contextos de 1024 tokens, lo que requiere corpora masivos (WebText tenía 40GB de texto).

**Conclusión:** El overlap es una forma de data augmentation implícita. En producción, GPT-2 usa `stride = max_length` (sin overlap) para evitar overfitting dado que el corpus es enorme. Para datasets pequeños, algo de overlap (ej. 50%) mejora la cobertura.


## 2.7 Creating token embeddings


In [ ]:
import torch

input_ids = torch.tensor([2, 3, 5, 1])

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

In [ ]:
print(embedding_layer(torch.tensor([3])))

In [ ]:
print(embedding_layer(input_ids))

## 2.8 Encoding word positions


In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

## Explicación 4 — Embeddings posicionales y su conexión con redes neuronales

### ¿Por qué los embeddings codifican significado?

Un embedding es simplemente una fila en la matriz de pesos de una capa nn.Embedding. Al inicio del entrenamiento, estos pesos son aleatorios — no significan nada. El significado emerge porque la red aprende que predecir correctamente la siguiente palabra requiere que tokens semánticamente similares tengan representaciones similares.

Matemáticamente: la capa de embedding es equivalente a un *one-hot encoding* seguido de una multiplicación por una matriz densa (sin sesgo, sin activación). El gradiente del error de predicción fluye hacia atrás por esta matriz, ajustando los vectores para minimizar la pérdida. Palabras que aparecen en contextos similares reciben gradientes similares -> convergen a regiones cercanas del espacio vectorial.

### Conexión con conceptos de redes neuronales

- Capa lineal: nn.Embedding es una capa lineal sin bias ni activación — solo una lookup table.
- Backpropagation: El gradiente actualiza solo las filas correspondientes a los tokens del batch, no toda la matriz.
- Representación latente: El embedding es la representación interna aprendida (hidden state inicial).
- Transfer learning: Los embeddings pre-entrenados (e.g., GPT-2) pueden reutilizarse en tareas downstream.

### Embeddings posicionales

La capa nn.Embedding por sí sola es *permutation invariant*: el vector del token "the" es idéntico sin importar si aparece en posición 1 o posición 100. Los positional embeddings corrigen esto: se suman al token embedding para que la red pueda distinguir "banco_posición_5" de "banco_posición_50".

El resultado final input_embeddings (forma [batch, seq_len, d_model]) es el tensor que entra al transformer
